# Kidaw'ida–Kiswahili Corpus Builder

Cleans the raw `waleghwa` parallel corpus CSV into one deduplicated `dav,swa` file.


## 1. Get the raw CSV

If the file isn't already on google collab,one will be prompted to upload the file 
(the `Kidaw'ida-Kiswahili-Parallel-Corpus.csv` from the `waleghwa` dataset).

In [ ]:
from pathlib import Path

RAW_FILENAME = "Kidaw'ida-Kiswahili-Parallel-Corpus.csv"

if not Path(RAW_FILENAME).exists():
    from google.colab import files
    print(f"Please upload {RAW_FILENAME}")
    uploaded = files.upload()
    RAW_FILENAME = list(uploaded.keys())[0]

print("Using file:", RAW_FILENAME)

## 2. Decode and load it

The file is otherwise plain ASCII but has a handful of stray Windows-1252 bytes
baked in (smart quotes, non-breaking spaces), so it's read as `cp1252` rather
than `utf-8` — `utf-8` would raise a decode error on those bytes.

In [ ]:
import csv

with open(RAW_FILENAME, encoding="cp1252", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = [row for row in reader if len(row) == 2]

print(f"Columns: {header}")
print(f"Loaded {len(rows)} raw rows")

## 3. Clean the text

Unicode-normalize, fix the mojibake characters, collapse extra whitespace.
We leave the `'` alone — it's used meaningfully in Kidaw'ida spelling
(`w'`, `ng'`, `m'`, etc.), not just as punctuation.

In [ ]:
import re
import unicodedata

# characters that show up as mojibake, mapped to their plain-ASCII equivalent
CHAR_FIXUPS = {
    "’": "'",
    "‘": "'",
    "“": '"',
    "”": '"',
    "\xa0": " ",   # non-breaking space
    "…": "...",
    "–": "-",
    "—": "-",
}
WHITESPACE = re.compile(r"\s+")

def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    for bad, good in CHAR_FIXUPS.items():
        text = text.replace(bad, good)
    return WHITESPACE.sub(" ", text).strip()

pairs = [(clean_text(dav), clean_text(swa)) for dav, swa in rows]
pairs[:5]

## 4. Drop empty rows and duplicates

In [ ]:
# drop rows where either side is empty after cleaning
pairs = [(dav, swa) for dav, swa in pairs if dav and swa]
print(f"{len(pairs)} rows left after dropping empty ones")

# drop exact duplicate (dav, swa) pairs, keeping the first occurrence
pairs = list(dict.fromkeys(pairs))
print(f"{len(pairs)} unique pairs after removing duplicates")

## 5. Save the corpus

In [ ]:
import json

OUT_CSV = "kidawida_kiswahili_corpus.csv"
OUT_JSONL = "kidawida_kiswahili_corpus.jsonl"

with open(OUT_CSV, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["dav", "swa"])
    writer.writerows(pairs)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for dav, swa in pairs:
        f.write(json.dumps({"dav": dav, "swa": swa}, ensure_ascii=False) + "\n")

print(f"Saved {len(pairs)} pairs to {OUT_CSV} and {OUT_JSONL}")

## Quick check

In [ ]:
for dav, swa in pairs[:5]:
    print(dav, "->", swa)

avg_dav = sum(len(d.split()) for d, s in pairs) / len(pairs)
avg_swa = sum(len(s.split()) for d, s in pairs) / len(pairs)
print(f"\nAverage length: {avg_dav:.1f} dav tokens, {avg_swa:.1f} swa tokens")

## Download the result (Colab only)

In [ ]:
try:
    from google.colab import files
    files.download(OUT_CSV)
    files.download(OUT_JSONL)
except ImportError:
    print("Not running in Colab — files saved locally:", OUT_CSV, "and", OUT_JSONL)